In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(r"D:\Project\Multi Agent\tmp\db\Object Counting\1039561247-preview.mp4")
assert cap.isOpened(), "Error reading video file"

# region_points = [(20, 400), (1080, 400)]                                      # line counting
# region_points = [[1244, 670], [1556, 675], [1618, 913], [1206, 905], [1247, 678]]  # rectangular region
region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360), (20, 400)]   # polygon region

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("object_counting_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize object counter object
counter = solutions.ObjectCounter(
    show=False,  # display the output
    region=region_points,  # pass region points
    model="yolo26n.pt",  # model="yolo26n-obb.pt" for object counting with OBB model.
    # classes=[0, 2],  # count specific classes, e.g., person and car with the COCO pretrained model.
    # tracker="botsort.yaml",  # choose trackers, e.g., "bytetrack.yaml"
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = counter(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

    # -- ĐƯA PHẦN NÀY VÀO BÊN TRONG VÒNG LẶP --
    cv2.imshow("Ultralytics Object Counting", results.plot_im) # Nên show results.plot_im thay vì im0 để thấy cả bounding box
    if cv2.waitKey(1) & 0xFF == ord('q'): # Nhấn phím 'q' để thoát an toàn
        break

# Các lệnh dọn dẹp giữ nguyên bên ngoài vòng lặp
cap.release()
video_writer.release()
cv2.destroyAllWindows() # Vẫn nên có dòng này để đóng cửa sổ khi kết thúc



Ultralytics Solutions:  {'source': None, 'model': 'yolo26n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': [(20, 400), (1080, 400), (1080, 360), (20, 360), (20, 400)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
WARNING No tracks found.
0: 1080x1920 3.3ms, 
Speed: 145.1ms track, 3.3ms solution per image at shape (1, 3, 1080, 1920)

WARNING No tracks found.
1: 1080x1920 4.0ms, 
Speed: 140.4ms track, 4.0ms solution per image at shape (1, 3, 1080, 1920)

WARNING No tracks found.
2: 1080x1920 1.4ms, 
Speed: 108

In [1]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")

results = model.predict(source="images (1).jpg", conf=0.25,save=True)
for r in results:
    print(r.boxes)


image 1/1 d:\Project\Multi Agent\tmp\db\Object Counting\images (1).jpg: 384x640 1 person, 1 car, 1 motorcycle, 121.9ms
Speed: 33.8ms preprocess, 121.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Results saved to D:\Project\Multi Agent\tmp\db\Object Counting\runs\detect\predict
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([2., 3., 0.])
conf: tensor([0.8724, 0.8453, 0.7810])
data: tensor([[151.8243,  35.1330, 295.5560, 112.3516,   0.8724,   2.0000],
        [ 24.5015,  46.9581, 109.9620, 160.7034,   0.8453,   3.0000],
        [ 23.6690,  16.0036, 108.7777, 143.0872,   0.7810,   0.0000]])
id: None
is_track: False
orig_shape: (168, 300)
shape: torch.Size([3, 6])
xywh: tensor([[223.6902,  73.7423, 143.7317,  77.2186],
        [ 67.2318, 103.8308,  85.4605, 113.7454],
        [ 66.2233,  79.5454,  85.1087, 127.0836]])
xywhn: tensor([[0.7456, 0.4389, 0.4791, 0.4596],
        [0.2241, 0.6180, 0.2849, 0.6771],
        [0.2207, 0.4735, 0.2837, 0.7

In [12]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


# 1. Khởi tạo model (Sửa lỗi chính tả: detection_model, confidence_threshold)
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo11n.pt",         # Thay bằng version bạn đang dùng, ví dụ yolo11n.pt
    confidence_threshold=0.25,      # Sửa: confidence_threshold (không phải confidense)
    device="cuda:0",                # "cuda" hoặc "cuda:0"
)

# 2. Chạy dự đoán cắt ảnh (Sửa lỗi cú pháp: dùng dấu '=' thay vì ':')
results = get_sliced_prediction(
    image="Ảnh chụp màn hình 2026-03-03 091245.png", # Sửa: image=
    detection_model=detection_model,                # Sửa: truyền đúng tên biến
    slice_height=256,
    slice_width=256,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

# 3. Xuất kết quả
results.export_visuals(export_dir="result/") # Lưu ảnh đã vẽ box vào thư mục result/

Performing prediction on 84 slices.


In [15]:
import cv2

In [31]:

from ultralytics import YOLO

model = YOLO("best.pt")
cap = cv2.VideoCapture(r"D:\Project\Multi Agent\tmp\db\Object Counting\istockphoto-981340772-640_adpp_is.mp4")

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("object_counting_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        break

    # 3. Chạy Tracking thay vì Predict để có ID vật thể (tốt cho việc đếm)
    # persist=True giúp model nhớ ID của vật thể giữa các frame
    results = model.track(im0, persist=True, conf=0.55)

    # 4. Vẽ Bounding Box và Label lên frame (Thay cho results.plot_im)
    # .plot() trả về một mảng numpy (BGR image) đã được vẽ sẵn
    annotated_frame = results[0].plot()

    # 5. Logic đếm thủ công (Ví dụ: đếm tổng số vật thể trong frame hiện tại)
    count_in_frame = len(results[0].boxes)
    
    # Ghi text số lượng lên ảnh
    cv2.putText(annotated_frame, f"Count: {count_in_frame}", (50, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # 6. Ghi video và hiển thị
    video_writer.write(annotated_frame)
    
    cv2.imshow("Custom Object Counting", annotated_frame) # Đã thêm tham số annotated_frame
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()


0: 384x640 16 Vehiculos, 88.1ms
Speed: 1.5ms preprocess, 88.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 Vehiculos, 98.2ms
Speed: 2.0ms preprocess, 98.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 Vehiculos, 89.7ms
Speed: 2.4ms preprocess, 89.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Vehiculos, 83.2ms
Speed: 1.8ms preprocess, 83.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Vehiculos, 89.9ms
Speed: 2.6ms preprocess, 89.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehiculos, 88.5ms
Speed: 2.4ms preprocess, 88.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 Vehiculos, 88.7ms
Speed: 2.0ms preprocess, 88.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 Vehiculos, 86.4ms
Speed: 1.9ms preprocess, 86.4ms inference, 1.0ms postprocess

In [39]:
import cv2
import supervision as sv
from ultralytics import YOLO

# Khởi tạo Model
model = YOLO("best.pt")

def callback(image_slice):
    results = model(image_slice, verbose=False)[0]
    return sv.Detections.from_ultralytics(results)

# Cấu hình Slicer với tham số đúng: overlap_wh
slicer = sv.InferenceSlicer(
    callback=callback,
    slice_wh=(640, 640), # Tăng size lên 640 để tận dụng VRAM của card RTX trên máy TUF
    overlap_wh=(128, 128),
    thread_workers=4
)

# Khởi tạo Annotators mới nhất
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

video_path = r"D:\Project\Multi Agent\tmp\db\Object Counting\istockphoto-981340772-640_adpp_is.mp4"
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Chạy Sliced Inference
    detections = slicer(frame)
    
    # Lọc bỏ các box quá nhỏ hoặc trùng lặp nếu cần
    # detections = detections.with_nms(threshold=0.5) 

    labels = [
        f"{model.names[class_id]} {confidence:.2f}"
        for class_id, confidence in zip(detections.class_id, detections.confidence)
    ]

    # Vẽ kết quả
    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    # Hiển thị số lượng
    cv2.putText(annotated_frame, f"Count: {len(detections)}", (40, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    cv2.imshow("Supervision Corrected", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

TypeError: InferenceSlicer.__init__() got an unexpected keyword argument 'overlap_wh_ratio'